In [1]:
import anndata
import sys
# sys.path.append("../../../../")
# sys.path.append("../../../../../")
sys.path.append("/archive/bioinformatics/DLLab/AixaAndrade/src/ARMED_genomics/utils")

from utils import read_adata
from splitter import DataSplitter
from model_train_utils import load_data
from config_split_paths import data_path,data2split_foldername,folder_splits
import glob
import anndata as ad

2024-04-03 22:40:20.246927: I tensorflow/stream_executor/platform/default/dso_loader.cc:48] Successfully opened dynamic library libcudart.so.10.1


In [2]:
# read paths
adata2split_path = data_path +"/"+folder_splits
print(adata2split_path)
splits_list = glob.glob(adata2split_path+"/*")
glob.glob(splits_list[0]+"/train/*")

/archive/bioinformatics/DLLab/AixaAndrade/data/Genomic_data/heart_data/Healthy_human_heart_data/log_transformed_3000hvggenes/splits


['/archive/bioinformatics/DLLab/AixaAndrade/data/Genomic_data/heart_data/Healthy_human_heart_data/log_transformed_3000hvggenes/splits/split_1/train/exprMatrix.npy',
 '/archive/bioinformatics/DLLab/AixaAndrade/data/Genomic_data/heart_data/Healthy_human_heart_data/log_transformed_3000hvggenes/splits/split_1/train/meta.csv',
 '/archive/bioinformatics/DLLab/AixaAndrade/data/Genomic_data/heart_data/Healthy_human_heart_data/log_transformed_3000hvggenes/splits/split_1/train/geneids.csv']

In [3]:
# get original donor data
adata2split_path_original = data_path + "/" + data2split_foldername
# X,var,obs = read_adata(adata2split_path_original)
X,var,obs = read_adata(adata2split_path_original, issparse=True)
adata = anndata.AnnData(X,obs=obs,var=var)

/archive/bioinformatics/DLLab/shared/CondaEnvironments/Aixa_ARMED_2/lib/python3.8/site-packages/anndata/_core/anndata.py:119: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


In [4]:
# Creating a dict with split paths and adata_splits
def get_splits_dict(splits_path,issparse=False):
    splits_list = glob.glob(splits_path+"/*")
    #print(splits_list)
    split_paths_dict = {}
    adata_splits_dict = {}
    for split in splits_list:
        split_name = split.split("/")[-1]
        split_paths_dict[split_name]={"train":split+"/train","val":split+"/val","test":split+"/test"}
        print(split_paths_dict[split_name])
        adata_splits_dict[split_name]=load_data(split_paths_dict[split_name], eval_test=True, scaling=None,issparse=issparse)
    return split_paths_dict,adata_splits_dict

In [5]:
def check_fold_leakage(adata_splits_dict):
    val_indices = []
    test_indices = []

    # Extracting indices for validation and test sets from each fold in the dictionary
    for split_key in adata_splits_dict.keys():
        fold_data = adata_splits_dict[split_key]
        val_indices.append(set(fold_data['val'].obs['original_index']))
        test_indices.append(set(fold_data['test'].obs['original_index']))

    # Checking for any overlaps (leakage) in validation and test sets
    num_folds = len(adata_splits_dict)
    overlap_detected = False
    for i in range(num_folds):
        for j in range(i + 1, num_folds):
            val_intersection = val_indices[i].intersection(val_indices[j])
            test_intersection = test_indices[i].intersection(test_indices[j])
            if val_intersection:
                print(f"Overlap detected in validation sets between folds {i} and {j}: {len(val_intersection)} common elements")
                overlap_detected = True
            if test_intersection:
                print(f"Overlap detected in test sets between folds {i} and {j}: {len(test_intersection)} common elements")
                overlap_detected = True

    if overlap_detected:
        return False
    else:
        print("No overlap detected in validation and test sets between folds.")
        return True


In [6]:
# Creating a dict with split paths and adata_splits
split_paths_dict,adata_splits_dict = get_splits_dict(data_path+"/"+folder_splits,issparse=True) 

{'train': '/archive/bioinformatics/DLLab/AixaAndrade/data/Genomic_data/heart_data/Healthy_human_heart_data/log_transformed_3000hvggenes/splits/split_1/train', 'val': '/archive/bioinformatics/DLLab/AixaAndrade/data/Genomic_data/heart_data/Healthy_human_heart_data/log_transformed_3000hvggenes/splits/split_1/val', 'test': '/archive/bioinformatics/DLLab/AixaAndrade/data/Genomic_data/heart_data/Healthy_human_heart_data/log_transformed_3000hvggenes/splits/split_1/test'}
{'train': '/archive/bioinformatics/DLLab/AixaAndrade/data/Genomic_data/heart_data/Healthy_human_heart_data/log_transformed_3000hvggenes/splits/split_2/train', 'val': '/archive/bioinformatics/DLLab/AixaAndrade/data/Genomic_data/heart_data/Healthy_human_heart_data/log_transformed_3000hvggenes/splits/split_2/val', 'test': '/archive/bioinformatics/DLLab/AixaAndrade/data/Genomic_data/heart_data/Healthy_human_heart_data/log_transformed_3000hvggenes/splits/split_2/test'}
{'train': '/archive/bioinformatics/DLLab/AixaAndrade/data/Geno

In [7]:
data_path+folder_splits

'/archive/bioinformatics/DLLab/AixaAndrade/data/Genomic_data/heart_dataHealthy_human_heart_data/log_transformed_3000hvggenes/splits'

In [8]:


# Check if there is any overlap between 
overlap_check_result = check_fold_leakage(adata_splits_dict)

No overlap detected in validation and test sets between folds.


In [9]:
# get train data from split 1
split_to_check = "split_5"
adata_train = adata_splits_dict[split_to_check]["train"]
print("adata_train shape",adata_train.shape)
# read test data from split 1
adata_test = adata_splits_dict[split_to_check]["test"]
print("adata_test shape",adata_test.shape)
# read val data from split 1
adata_val = adata_splits_dict[split_to_check]["val"]
print("adata_val shape",adata_val.shape)

adata_train shape (291681, 3000)
adata_test shape (97226, 3000)
adata_val shape (97227, 3000)


In [10]:
adata_test.obs

,Unnamed: 0,Unnamed: 0.1,Age,AgeBin,DeathType,DonorID,Gender,Organ,Race,SampleType,Source,Tissue,TissueDetail,_index,celltype,protocol,sampleID,n_genes,batch,original_index
0,106505,106505,NaN,50-55,DBD(brain death),H2,Male,Heart,Caucasian,normal,Nuclei,LV,left ventricle,b'CCGAACGAGCTGTTCA-1-H0026_LV_V3',Ventricular_Cardiomyocyte,10X3'V3,H0026_LV_V3,967,H0026_LV_V3,106505
1,473167,473167,NaN,60-65,DCD(circulatory death),D11,Female,Heart,Caucasian,normal,CD45+,SP,septum,b'TGATTTCAGTAGAGTT-1-HCAHeart8102861',Myeloid,10X3'V3,HCAHeart8102861,2112,HCAHeart8102861,473167
2,324198,324198,NaN,65-70,DCD(circulatory death),D6,Male,Heart,Caucasian,normal,Nuclei,AX,apex,b'AACTCAGTCACGAAGG-1-HCAHeart7888927',NotAssigned,10X3'V2,HCAHeart7888927,1272,HCAHeart7888927,324198
3,331762,331762,NaN,60-65,DCD(circulatory death),D7,Male,Heart,Caucasian,normal,Nuclei,RA,right atrium,b'CAACCAATCAGTTGAC-1-HCAHeart7888923',Endothelial,10X3'V2,HCAHeart7888923,481,HCAHeart7888923,331762
4,7505,7505,NaN,50-55,DBD(brain death),H5,Female,Heart,Caucasian,Pulmonary disease,Nuclei,AX,apex,b'TCGGGCACAGTAGTTC-1-H0015_apex',Ventricular_Cardiomyocyte,10X3'V3,H0015_apex,589,H0015_apex,7505
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
97221,346622,346622,NaN,60-65,DCD(circulatory death),D11,Female,Heart,Caucasian,normal,Nuclei,LA,left atrium,b'CTCACTGGTACCAATC-1-HCAHeart8287123',Pericytes,10X3'V3,HCAHeart8287123,646,HCAHeart8287123,346622
97222,327069,327069,NaN,65-70,DCD(circulatory death),D6,Male,Heart,Caucasian,normal,Nuclei,AX,apex,b'GTTACAGGTAGAGCTG-1-HCAHeart7888927',Myeloid,10X3'V2,HCAHeart7888927,752,HCAHeart7888927,327069
97223,374871,374871,NaN,65-70,DCD(circulatory death),D6,Male,Heart,Caucasian,normal,Cells,LV,left ventricle,b'TCACACCAGGTAGCCA-1-HCAHeart7905327',Endothelial,10X3'V3,HCAHeart7905327,446,HCAHeart7905327,374871
97224,137337,137337,NaN,45-50,DBD(brain death),H7,Female,Heart,Caucasian,normal,Nuclei,LV,left ventricle,b'TTCGGTCCAAGAGCTG-1-H0035_LV',Endothelial,10X3'V3,H0035_LV,1012,H0035_LV,137337


In [11]:
# check that train, test and val should sum up total data
adata_train.shape[0]+adata_val.shape[0]+adata_test.shape[0]

486134

In [12]:
import numpy as np
# # check all donors list
adata.obs['batch'] = adata.obs["sampleID"].astype('category')
donor_list = list(np.unique(adata.obs["batch"]))
# Seen donor list
donor_seen_list = list(np.unique(adata_train.obs["batch"]))
print(donor_seen_list)

['H0015_LA_new', 'H0015_LV', 'H0015_RA', 'H0015_RV', 'H0015_apex', 'H0015_septum', 'H0020_LA_new', 'H0020_LV', 'H0020_RA', 'H0020_RV', 'H0020_apex', 'H0020_septum', 'H0025_LA', 'H0025_LV', 'H0025_RA', 'H0025_RV', 'H0025_apex', 'H0025_septum', 'H0026_LA', 'H0026_LV_V3', 'H0026_RA', 'H0026_RV', 'H0026_apex', 'H0026_septum2', 'H0035_LA', 'H0035_LV', 'H0035_RA', 'H0035_RV', 'H0035_apex', 'H0035_septum', 'H0037_Apex', 'H0037_LA_corr', 'H0037_LV', 'H0037_RA_corr', 'H0037_RV', 'H0037_septum', 'HCAHeart7606896', 'HCAHeart7656534', 'HCAHeart7656535', 'HCAHeart7656536', 'HCAHeart7656537', 'HCAHeart7656538', 'HCAHeart7656539', 'HCAHeart7664652', 'HCAHeart7664653', 'HCAHeart7664654', 'HCAHeart7698015', 'HCAHeart7698016', 'HCAHeart7698017', 'HCAHeart7702873', 'HCAHeart7702874', 'HCAHeart7702875', 'HCAHeart7702876', 'HCAHeart7702877', 'HCAHeart7702878', 'HCAHeart7702879', 'HCAHeart7702880', 'HCAHeart7702881', 'HCAHeart7702882', 'HCAHeart7728604', 'HCAHeart7728605', 'HCAHeart7728606', 'HCAHeart772860

In [13]:
# check if the stratification worked

# call the splitter stratification test
splitter = DataSplitter()
comp_df = splitter.check_stratification(adata, adata_train, adata_val, adata_test, ['batch', 'celltype'])
# add donor and celltype columns
comp_df["batch"] = [x.split("_")[0] for x in comp_df.index ]
comp_df["celltype"] = [x.split("_")[1] for x in comp_df.index ]


# Donor is in seen list
condition1 = comp_df["batch"].isin(donor_seen_list)
# Values equal to zero
condition2 = comp_df[['Train', 'Validation', 'Test']].eq(0).any(axis=1)
filtered_df = comp_df.loc[condition1 & condition2]
# We want to see the seen donors well distributed trough the train, test and validation datasets. No zero entries.


In [14]:
filtered_df

,Original,Train,Validation,Test,batch,celltype
HCAHeart7606896_Myeloid,0.000004,0.000007,0.00000,0.00000,HCAHeart7606896,Myeloid
HCAHeart7606896_NotAssigned,0.000006,0.000007,0.00000,0.00001,HCAHeart7606896,NotAssigned
HCAHeart7656534_Neuronal,0.000002,0.000003,0.00000,0.00000,HCAHeart7656534,Neuronal
HCAHeart7656535_Fibroblast,0.000004,0.000007,0.00000,0.00000,HCAHeart7656535,Fibroblast
HCAHeart7656535_Lymphoid,0.000004,0.000003,0.00000,0.00001,HCAHeart7656535,Lymphoid
...,...,...,...,...,...,...
HCAHeart8287125_Lymphoid,0.000008,0.000010,0.00001,0.00000,HCAHeart8287125,Lymphoid
HCAHeart8287125_Neuronal,0.000006,0.000010,0.00000,0.00000,HCAHeart8287125,Neuronal
HCAHeart8287126_doublets,0.000002,0.000003,0.00000,0.00000,HCAHeart8287126,doublets
HCAHeart8287127_doublets,0.000002,0.000000,0.00000,0.00001,HCAHeart8287127,doublets


In [15]:
adata_val

AnnData object with n_obs × n_vars = 97227 × 3000
    obs: 'Unnamed: 0', 'Unnamed: 0.1', 'Age', 'AgeBin', 'DeathType', 'DonorID', 'Gender', 'Organ', 'Race', 'SampleType', 'Source', 'Tissue', 'TissueDetail', '_index', 'celltype', 'protocol', 'sampleID', 'n_genes', 'batch', 'original_index', 'combined_stratify_col'
    var: 'Unnamed: 0'

In [16]:
adata_train.obs["combined_stratify_col"]

0                           H0037_LV_Pericytes
1                      H0037_RA_corr_Pericytes
2         HCAHeart7888929_Atrial_Cardiomyocyte
3                    HCAHeart7844004_Pericytes
4                    HCAHeart7829977_Pericytes
                          ...                 
291676             HCAHeart7905332_Endothelial
291677                      H0025_RV_Pericytes
291678      H0020_LV_Ventricular_Cardiomyocyte
291679             HCAHeart7702875_Endothelial
291680                      H0035_LV_Pericytes
Name: combined_stratify_col, Length: 291681, dtype: object

In [17]:
adata_train.obs["combined_stratify_col"]

0                           H0037_LV_Pericytes
1                      H0037_RA_corr_Pericytes
2         HCAHeart7888929_Atrial_Cardiomyocyte
3                    HCAHeart7844004_Pericytes
4                    HCAHeart7829977_Pericytes
                          ...                 
291676             HCAHeart7905332_Endothelial
291677                      H0025_RV_Pericytes
291678      H0020_LV_Ventricular_Cardiomyocyte
291679             HCAHeart7702875_Endothelial
291680                      H0035_LV_Pericytes
Name: combined_stratify_col, Length: 291681, dtype: object

In [18]:
# from matplotlib import pyplot as plt
# index_val = adata_val.obs.loc[adata_val.obs['combined_stratify_col']=='5163_Neu-mat'].index
# print("index_val.shape",index_val.shape)
# plt.hist(adata_val.X[np.array(index_val.astype(int)),:])


In [19]:
adata_train.obs

,Unnamed: 0,Unnamed: 0.1,Age,AgeBin,DeathType,DonorID,Gender,Organ,Race,SampleType,...,Tissue,TissueDetail,_index,celltype,protocol,sampleID,n_genes,batch,original_index,combined_stratify_col
0,163906,163906,NaN,55-60,DBD(brain death),H4,Male,Heart,Caucasian,normal,...,LV,left ventricle,b'GTGGAAGAGTCATACC-1-H0037_LV',Pericytes,10X3'V3,H0037_LV,886,H0037_LV,163906,H0037_LV_Pericytes
1,157467,157467,NaN,55-60,DBD(brain death),H4,Male,Heart,Caucasian,normal,...,RA,right atrium,b'CTCAAGATCCTAAGTG-1-H0037_RA_corr',Pericytes,10X3'V3,H0037_RA_corr,1160,H0037_RA_corr,157467,H0037_RA_corr_Pericytes
2,344346,344346,NaN,60-65,DCD(circulatory death),D7,Male,Heart,Caucasian,normal,...,LA,left atrium,b'CGCGTTTTCCCGGATG-1-HCAHeart7888929',Atrial_Cardiomyocyte,10X3'V2,HCAHeart7888929,1341,HCAHeart7888929,344346,HCAHeart7888929_Atrial_Cardiomyocyte
3,441635,441635,NaN,65-70,DCD(circulatory death),D6,Male,Heart,Caucasian,normal,...,AX,apex,b'GATGCTAGTGCGGTAA-1-HCAHeart7844004',Pericytes,10X3'V2,HCAHeart7844004,1434,HCAHeart7844004,441635,HCAHeart7844004_Pericytes
4,249172,249172,NaN,55-60,DBD(brain death),D3,Male,Heart,Caucasian,normal,...,AX,apex,b'AGATCTGTCGCGATCG-1-HCAHeart7829977',Pericytes,10X3'V2,HCAHeart7829977,529,HCAHeart7829977,249172,HCAHeart7829977_Pericytes
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
291676,388468,388468,NaN,65-70,DCD(circulatory death),D6,Male,Heart,Caucasian,normal,...,AX,apex,b'ATTCCTACATGTCGTA-1-HCAHeart7905332',Endothelial,10X3'V3,HCAHeart7905332,1131,HCAHeart7905332,388468,HCAHeart7905332_Endothelial
291677,87498,87498,NaN,50-55,DBD(brain death),H3,Male,Heart,Asian,normal,...,RV,right ventricle,b'AGGGAGTGTTACTCAG-1-H0025_RV',Pericytes,10X3'V3,H0025_RV,473,H0025_RV,87498,H0025_RV_Pericytes
291678,54886,54886,NaN,40-45,DBD(brain death),H6,Female,Heart,Caucasian,normal,...,LV,left ventricle,b'TGCCGAGTCGTGAGAG-1-H0020_LV',Ventricular_Cardiomyocyte,10X3'V3,H0020_LV,527,H0020_LV,54886,H0020_LV_Ventricular_Cardiomyocyte
291679,207892,207892,NaN,60-65,DCD(circulatory death),D2,Male,Heart,Caucasian,normal,...,RV,right ventricle,b'ACGGAGACAGGTTTCA-1-HCAHeart7702875',Endothelial,10X3'V2,HCAHeart7702875,799,HCAHeart7702875,207892,HCAHeart7702875_Endothelial
